In [ ]:
#Quickstart from the docs

In [1]:
import torch 
from torch import nn 
from torch.utils.data import DataLoader
from torchvision import datasets 
from torchvision.transforms import v2

In [5]:
#Downloading training data from open datasets 
training_data = datasets.FashionMNIST(
	root="data",
	train="True",
	download="True",
	transform=v2.Compose([v2.ToImage(),v2.ToDtype(torch.float32, scale=True)])
)

test_data = datasets.FashionMNIST(
	root="data",
	train="False",
	download="True",
	transform=v2.Compose([v2.ToImage(),v2.ToDtype(torch.float32, scale=True)])
	)

In [6]:
#Define a batch size of 64 features and labels 
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break
    

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


In [7]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

Using cuda device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [8]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [9]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [10]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [11]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.301853  [   64/60000]
loss: 2.288743  [ 6464/60000]
loss: 2.269431  [12864/60000]
loss: 2.264127  [19264/60000]
loss: 2.248548  [25664/60000]
loss: 2.218840  [32064/60000]
loss: 2.238500  [38464/60000]
loss: 2.202988  [44864/60000]
loss: 2.201139  [51264/60000]
loss: 2.177127  [57664/60000]
Test Error: 
 Accuracy: 40.2%, Avg loss: 2.162758 

Epoch 2
-------------------------------
loss: 2.176737  [   64/60000]
loss: 2.165530  [ 6464/60000]
loss: 2.104930  [12864/60000]
loss: 2.118814  [19264/60000]
loss: 2.073964  [25664/60000]
loss: 2.006920  [32064/60000]
loss: 2.043751  [38464/60000]
loss: 1.960992  [44864/60000]
loss: 1.970184  [51264/60000]
loss: 1.902891  [57664/60000]
Test Error: 
 Accuracy: 57.4%, Avg loss: 1.890302 

Epoch 3
-------------------------------
loss: 1.927183  [   64/60000]
loss: 1.902999  [ 6464/60000]
loss: 1.775582  [12864/60000]
loss: 1.813659  [19264/60000]
loss: 1.714675  [25664/60000]
loss: 1.647555  [32064/600